In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from deap import base, creator, tools, algorithms
import random

# Load the dataset
# Assuming the historical stock data is in a CSV file with columns like: 'Date', 'Open', 'High', 'Low', 'Close', 'Volume'
data = pd.read_csv('historical_stock_data.csv')

# Feature engineering
data['Price_Change'] = data['Close'].diff()
data['SMA_5'] = data['Close'].rolling(window=5).mean()
data['SMA_10'] = data['Close'].rolling(window=10).mean()
data['Volatility'] = data['Close'].rolling(window=5).std()
data['Return'] = data['Close'].pct_change()

# Target variable
data['Target'] = np.where(data['Close'].shift(-1) > data['Close'], 1, 0)

# Drop NaN values generated by feature engineering
data = data.dropna()

# Features and target selection
features = ['Open', 'High', 'Low', 'Close', 'Volume', 'Price_Change', 'SMA_5', 'SMA_10', 'Volatility', 'Return']
target = 'Target'

X = data[features]
y = data[target]

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Genetic Algorithm to optimize Random Forest Classifier
creator.create('FitnessMax', base.Fitness, weights=(1.0,))
creator.create('Individual', list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()

# Attribute generator
toolbox.register('n_estimators', random.randint, 10, 200)
toolbox.register('max_depth', random.randint, 1, 20)
toolbox.register('min_samples_split', random.uniform, 0.1, 1.0)

toolbox.register('individual', tools.initCycle, creator.Individual,
                 (toolbox.n_estimators, toolbox.max_depth, toolbox.min_samples_split), n=1)

toolbox.register('population', tools.initPopulation, list, toolbox.individual)

# Evaluation function
def evaluate_individual(individual):
    n_estimators, max_depth, min_samples_split = individual
    rf_classifier = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=42
    )
    rf_classifier.fit(X_train, y_train)
    y_pred = rf_classifier.predict(X_test)
    return accuracy_score(y_test, y_pred),

toolbox.register('mate', tools.cxTwoPoint)
toolbox.register('mutate', tools.mutPolynomialBounded, low=[10, 1, 0.1], up=[200, 20, 1.0], indpb=0.2, eta=1.0)
toolbox.register('select', tools.selTournament, tournsize=3)
toolbox.register('evaluate', evaluate_individual)

# Genetic Algorithm setup
population = toolbox.population(n=20)
NGEN = 10
CXPB, MUTPB = 0.5, 0.2

for gen in range(NGEN):
    offspring = algorithms.varAnd(population, toolbox, cxpb=CXPB, mutpb=MUTPB)
    fits = list(map(toolbox.evaluate, offspring))
    for fit, ind in zip(fits, offspring):
        ind.fitness.values = fit
    population = toolbox.select(offspring, k=len(population))

# Get the best individual
best_individual = tools.selBest(population, k=1)[0]
n_estimators, max_depth, min_samples_split = best_individual

# Train the optimized RandomForestClassifier
rf_classifier = RandomForestClassifier(
    n_estimators=n_estimators,
    max_depth=max_depth,
    min_samples_split=min_samples_split,
    random_state=42
)
rf_classifier.fit(X_train, y_train)

# Make predictions
y_pred = rf_classifier.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f'Optimized Accuracy: {accuracy:.2f}')
print(classification_report(y_test, y_pred))

# Feature Importance
importances = rf_classifier.feature_importances_
feature_importances = pd.DataFrame({'Feature': features, 'Importance': importances}).sort_values(by='Importance', ascending=False)
print(feature_importances)

# Predict future movement (example for the next t time)
latest_data = data[features].iloc[-1:].values
future_prediction = rf_classifier.predict(latest_data)
print('Prediction for next time step:', 'Up' if future_prediction[0] == 1 else 'Down')
